# Week 2 Lesson Notebook: Word2Vec_Embeddings & GPT-2 Predictions

In this notebook, we play with some classic word embeddings (using Word2Vec) and then use an old Language Model, GPT-2, to make a few next-word predictions. The purpose is start building up some intuition for the entities and concepts we are working with.  We use "embedding" vectors to represent the words in language as we process them in neural networks.  Embeddings are a fuzzy representation of words.  We use decoder transformers to predict the next word based on the previous sequence of words.  We'll see the mechanics of feeding a sequence of words into a transformer to predict the next word.  We'll use this process through out the rest of the class.<br>

**Note:** In this and other lesson notebooks we will also pose questions for you to think about and solve, if you are interested. Look for '**Additional Question**'.

## 1. Setup

This notebook requires the tensorflow dataset and other prerequisites that you must download and then store locally.

In [1]:
!pip install gensim --quiet
!pip install pydot --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 41.2 MB/s eta 0:00:00


In [2]:
import sklearn as sk
import os
import nltk
from nltk.corpus import reuters
from nltk.data import find

import matplotlib.pyplot as plt

import re

import gensim

import numpy as np

In [3]:
#In case we want to know our installed transformers library version
!pip list | grep gensim
!pip list | grep numpy

gensim                                   4.4.0
numpy                                    2.0.2


Below is a helper function for similarity evaluation:

In [4]:
# We are using cosine similarity

def cos_sim(a, b):

    """
    Computes the cosine similarity
    """
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)


## 2. Word Embeddings

Next, we get the word2vec model from nltk.

In [5]:
nltk.download('word2vec_sample')

word2vec_sample = str(find('models/word2vec_sample/pruned.word2vec.txt'))
model = gensim.models.KeyedVectors.load_word2vec_format(word2vec_sample, binary=False)

[nltk_data] Downloading package word2vec_sample to /root/nltk_data...
[nltk_data]   Unzipping models/word2vec_sample.zip.


How many words are in the vocabulary?

In [6]:
len(model.key_to_index)

43981

How do the word vectors look like? As expected:

In [7]:
model['school']

array([ 3.70471e-02,  1.14410e-02,  1.49575e-02,  8.87547e-02,
        3.96226e-02, -2.67453e-02,  6.33962e-02, -1.90189e-02,
       -1.89446e-03, -3.68490e-02,  1.01038e-01,  1.85236e-02,
        2.69434e-02, -4.00188e-02, -4.29905e-02,  4.31887e-02,
       -8.12264e-02,  5.72052e-03,  5.54717e-02, -3.56604e-02,
        8.32075e-02,  6.93396e-02,  4.72995e-03,  6.97358e-02,
        1.96875e-03, -1.41849e-01,  9.22464e-04,  7.48867e-02,
        4.85377e-02, -1.02028e-02,  4.14056e-02, -4.33868e-02,
        1.62453e-02,  3.04599e-03, -6.61698e-02, -6.06226e-02,
        9.27169e-02, -2.04056e-02,  1.88207e-02,  5.07170e-02,
        5.29953e-03,  5.19056e-02,  4.47736e-02, -2.05047e-02,
        1.39670e-02,  5.86415e-02,  6.97358e-02, -1.12924e-02,
       -4.49717e-02,  9.31132e-02, -4.75471e-02, -4.95283e-02,
       -1.44251e-03, -4.61604e-02,  8.59811e-02, -8.47924e-02,
       -4.23962e-02,  1.78302e-02, -5.00236e-03, -6.45849e-02,
       -3.58585e-02, -1.62453e-02,  4.31887e-02, -2.060

Let's vectorize at a few words and look at the cosine similarities:

In [8]:
vec_car = model['car']
vec_vehicle = model['vehicle']
vec_school = model['school']

In [9]:
cos_sim(vec_car, vec_school)

np.float32(0.109525874)

In [10]:
cos_sim(vec_car, vec_vehicle)

np.float32(0.7821095)

In [11]:
cos_sim(vec_school, vec_vehicle)

np.float32(0.09002902)

Let's play with a few more examples...

In [12]:
vec_related = model['automotive']
cos_sim(vec_car, vec_related)

np.float32(0.37437758)

In [13]:
len(model)

43981

In [14]:
vec_unrelated = model['aardvark']
cos_sim(vec_car, vec_unrelated)

KeyError: "Key 'aardvark' not present"

Oops! Out of vocabulary used to be a real issue for classic word embeddings.

**Additional Question 1:** Can you verify that the word vectors represent interesting syntactic and semantic relationships well, like '*run* is to *running* as *swim* is *swimming*'. How could you approach that? (Hint: conceptually, 'ing' ~ 'running' - 'run ).

## 3. Simple Next-Word Predictions with GPT-2

We will now download the GPT2 model from Huggingface and use it to get a feeling for these next-word predictions


In [ ]:
#!pip install transformers  --quiet

In [15]:
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

In [16]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The model requires tokenized input. I.e., each word is split into tokens (one word can be comprised of one or more tokens) and the token id is used as the input to the model:

In [17]:
inputs = tokenizer("Today is a very nice", return_tensors="pt")

In [18]:
inputs

{'input_ids': tensor([[8888,  318,  257,  845, 3621]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

We see the five input ids and the corresponding 'attention_masks' (~'should' the model pay attention to the position?').

Now we apply the model to the input:

In [19]:
output = model(**inputs)

In [20]:
len(output)

2

Why '2'? The [Huggingface documentation ](https://huggingface.co/docs/transformers/model_doc/gpt2#transformers.GPT2LMHeadModel)is very helpful.

In [21]:
output.keys()

odict_keys(['logits', 'past_key_values'])

In [22]:
output.logits.shape

torch.Size([1, 5, 50257])

What could be the meaning of these dimensions?

Ok, let's the positions by the logits:

In [23]:
logits_last_position = (output.logits.detach()[0, -1])
np.argsort(logits_last_position)

tensor([31573,   208,   214,  ...,   290,   640,  1110])

What is the token corresponding to the highest logit?

In [24]:
tokenizer.decode([1110])

' day'

Does this look right? It does...

What are the corresponding *relative* probabilities of the 2 most common words?

In [25]:
torch.exp(logits_last_position[1110])/ torch.exp(logits_last_position[640])

tensor(8.6250)

Substantially more likely to pick token 1.  What was token 2?

In [26]:
tokenizer.decode([640])

' time'

'Today is a very nice **day**' vs 'Today is a very nice **time**'. Makes sense...

**Additional Question 2:** How could you possibly use a language model to determine whether 'This was fun' has *positive* or *negative* sentiment? (Note, GPT-2 isn't that great to say the least, but the principle is instructive.)
